# Feeding / Non-Feeding Model Catalogue & Gold-Standard Evaluation

Two things:

1. **Catalogues every `.keras` model** in `feeding_model/` (both `models/` and the older
   `data/Final_Feeding_Images/` folder) with date, size, and what was varied when it was trained.
2. **Evaluates each model against two truth sets** and reports **accuracy, macro-F1, and per-class
   recall** for each, side by side, so you have one documented comparison table.

### The two truth sets
| key | file | images | Feeding / Non-feeding | why |
|---|---|---|---|---|
| `gold` | `test_data/Joint_gold_standard/Joint_gold_st_with_NF.csv` | ~4,414 | 2,145 / 2,269 (**48.6% / 51.4%**) | **balanced** → primary; accuracy is trustworthy |
| `bimby2024` | `test_data/2024_BIMBY_appended/BC2024_goldstandard_appendfilenames2.csv` | ~6,156 | 3,908 / 2,248 (**63.5% / 36.5%**) | larger, **imbalanced** → read per-class recall, not accuracy |

See section 3 for the full note on why the imbalance matters. The two sets overlap in images (the
balanced set draws partly from the same `2024_BIMBY_appended` folder), so the second set's value is
its **size** and its **native F/NF labelling**, not statistical independence from the first.

**How to run:** open from the `feeding_model/` directory, select your `butterflyai` kernel, Run All.
No internet needed. ~27 unique models × ~10k images — minutes on GPU, longer on CPU. Nothing is retrained.

**Label convention:** models output a sigmoid where **0 = Feeding, 1 = Non-feeding** (alphabetical, as in
training). Each truth set is mapped the same way. Any model scoring < 0.5 has its predictions flipped and
is flagged (`label_flip`).

> ⚠️ The 2024 `detect*` models were trained on detectron-**cropped** butterflies. Scored here on **uncropped**
> photos they are off their native input — read low scores as 'not comparable', not 'bad model'.

## 1. Configuration & paths

In [ ]:
from pathlib import Path
import os, json, time
# Your project env pins tf-keras (Keras 2), which is how these models were trained/saved.
# If a model later fails to load with a Keras-version error, uncomment the next line and restart:
# os.environ['TF_USE_LEGACY_KERAS'] = '1'   # route tensorflow.keras -> Keras 2 (tf-keras)
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import load_model
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             precision_recall_fscore_support)

# --- locate the feeding_model directory (run this notebook from there) ---
MODULE_DIR = Path.cwd()
if (MODULE_DIR / 'feeding_model' / 'models').is_dir():
    MODULE_DIR = MODULE_DIR / 'feeding_model'
assert (MODULE_DIR / 'models').is_dir(), f'Set MODULE_DIR by hand — got {MODULE_DIR}'
REPO_ROOT       = MODULE_DIR.parent
MODELS_DIR      = MODULE_DIR / 'models'
DATA_MODELS_DIR = MODULE_DIR / 'data' / 'Final_Feeding_Images'
RESULTS_DIR     = MODULE_DIR / 'results'; RESULTS_DIR.mkdir(exist_ok=True)
TEST_DATA       = REPO_ROOT / 'test_data'

IMG_SIZE  = (224, 224)
BATCH     = 64
THRESHOLD = 0.5
Y_TO_LABEL = {0: 'Feeding', 1: 'Non-feeding'}   # sigmoid convention: 0 = Feeding

# --- the two truth sets. label_map sends each file's raw label to 0=Feeding / 1=Non-feeding ---
TRUTH_SETS = [
    {'key': 'gold',
     'name': 'Balanced gold (Joint_gold_st_with_NF)',
     'csv': TEST_DATA / 'Joint_gold_standard' / 'Joint_gold_st_with_NF.csv',
     'file_col': 'FileName', 'label_col': 'field.dsf.butterfly.activity',
     'label_map': {'Feeding': 0, 'Nonfeeding': 1},
     'image_dirs': [TEST_DATA / 'Joint_gold_standard', TEST_DATA / '2024_BIMBY_appended']},
    {'key': 'bimby2024',
     'name': '2024 BIMBY appended (larger, imbalanced)',
     'csv': TEST_DATA / '2024_BIMBY_appended' / 'BC2024_goldstandard_appendfilenames2.csv',
     'file_col': 'FileName', 'label_col': 'feeding_status',
     'label_map': {'F': 0, 'NF': 1},
     'image_dirs': [TEST_DATA / '2024_BIMBY_appended']},
]
PRIMARY_KEY = 'gold'   # table is ranked by this set's macro-F1

print('feeding_model dir :', MODULE_DIR)
for ts in TRUTH_SETS:
    print(f"  truth set '{ts['key']}' CSV exists: {ts['csv'].is_file()}")
print('TF version        :', tf.__version__)

## 2. Static catalogue of every model file

Built from what is on disk plus what we know about each run. `documented_result` = numbers already captured
in the repo's `results/`; blank = never recorded.

In [ ]:
def model_meta(name):
    exact = {
        'FINAL_Real_Feeding_unfrozen_xval_augmented.keras': (
            '2025 final', 'real+super, unfrozen conv5, 5-fold xval checkpoint (best val_loss)', ''),
        'Final_feeding_model_ALL.keras': (
            '2025 final', 'all images (real+super), head-only (frozen base)', ''),
        'Final_feeding_model_ALL_unfrozen.keras': (
            '2025 final', 'all images (real+super), unfrozen conv5', ''),
        'REAL_AND_SUPER_Final_feeding_model_unfrozen.keras': (
            '2025 final (PRODUCTION)', 'real+super, unfrozen conv5 — the model the monarch pipeline loads',
            'BIMBY-2024 gold: acc 85.4%, macro-F1 0.850 (the "augmented" run)'),
        'REAL_ONLY_Final_feeding_model_unfrozen.keras': (
            '2025 final', 'real images only, unfrozen conv5',
            'BIMBY-2024 gold: acc 80.4%, macro-F1 ~0.80 (the "baseline" run)'),
        'Real_Feeding_frozen_xval.keras': (
            '2025 May', 'real only, frozen base (head only), 5-fold xval', ''),
        'Real_Feeding_unfrozen_noxval.keras': (
            '2025 May', 'real only, unfrozen conv5, single 80/20 split', ''),
        'Real_Feeding_unfrozen_xval.keras': (
            '2025 May', 'real only, unfrozen conv5, 5-fold xval', ''),
        'ResNet50_data_redo1.keras': ('2024 Sep', 'baseline re-run on the cleaned/redo dataset', ''),
        'ResNet50_detectnone.keras': ('2024 Oct', 'trained on UNcropped images (detectron off)', ''),
        'ResNet50_detectall.keras': ('2024 Oct', 'trained on ALL detectron-cropped butterflies', '⚠ cropped-input model'),
        'ResNet50_detect500_test.keras': ('2024 Oct', 'detectron-cropped, 500 tier (test variant)', '⚠ cropped-input model'),
    }
    n = name
    if n in exact: return exact[n]
    if n.startswith('ResNet50_data_redo_squeeze'):
        v = n.replace('ResNet50_data_redo_squeeze','').replace('.keras','')
        return ('2024 Sep', f'dense-head architecture iteration ("squeeze" v{v})', '')
    if n.startswith('ResNet50_data_squeeze_l2'):
        return ('2024 Sep', 'squeeze head + L2 weight regularization', '')
    if n.startswith('ResNet50_detectall_kfold'):
        return ('2024 Oct', 'detectron-cropped, cross-validated', '⚠ cropped-input model')
    if n.startswith('ResNet50_detect'):
        tier = ''.join(ch for ch in n if ch.isdigit())
        return ('2024 Oct', f'trained on detectron-cropped butterflies (tier {tier})', '⚠ cropped-input model')
    return ('unknown', '(uncategorised)', '')

rows, seen = [], set()
for folder in [MODELS_DIR, DATA_MODELS_DIR]:
    for p in sorted(folder.glob('*.keras')):
        fam, varied, doc = model_meta(p.name)
        dup = p.name in seen; seen.add(p.name)
        rows.append({'file': p.name, 'folder': folder.name, 'trained': fam,
            'date': time.strftime('%Y-%m-%d', time.localtime(p.stat().st_mtime)),
            'size_mb': round(p.stat().st_size/1e6, 1), 'what_varied': varied,
            'documented_result': doc, 'duplicate_of_models_copy': dup, 'path': str(p)})
cat_df = pd.DataFrame(rows)
print(f'{len(cat_df)} files on disk; {cat_df["file"].nunique()} unique')
pd.set_option('display.max_colwidth', 60)
cat_df[['file','folder','trained','date','size_mb','what_varied','documented_result','duplicate_of_models_copy']]

## 3. Truth sets — and exactly how the second one is unbalanced

**`gold` — balanced (primary).** `Joint_gold_st_with_NF.csv`: **2,145 Feeding (48.6%) / 2,269 Non-feeding
(51.4%)** — essentially 50/50. On a balanced set, plain **accuracy is meaningful**: a model can't score well
by leaning on one class, so accuracy, macro-F1 and the two recalls all tell a consistent story.

**`bimby2024` — imbalanced (secondary).** `BC2024_goldstandard_appendfilenames2.csv`: **3,908 Feeding (63.5%)
/ 2,248 Non-feeding (36.5%)** — feeding outnumbers non-feeding by about **1.74 to 1**. Why that matters:

- A trivial model that predicts **"Feeding" for every image** scores **63.5% accuracy** on this set while
  catching **0%** of non-feeding photos. So a headline accuracy in the 60s here can mean *nothing was learned*.
- Accuracy is therefore **not** a fair yardstick on this set. Judge models by **`bimby2024_nonfeed_recall`**
  (did it actually catch the minority class?) and **`bimby2024_macro_f1`** (which averages the two classes
  evenly, so the large feeding class can't paper over failure on non-feeding).
- Expect every model's **accuracy to look a few points higher on `bimby2024` than on `gold`** purely because
  the easy majority class is over-represented — that gap is the imbalance talking, not a better model.

The next cell prints the exact class counts, percentages, and the always-Feeding baseline accuracy for each set
at runtime, from the images actually found on disk.

In [ ]:
def build_index(dirs):
    idx = {}
    for d in dirs:
        if d.is_dir():
            for fn in os.listdir(d):
                idx.setdefault(fn, str(d / fn))
    return idx

for ts in TRUTH_SETS:
    raw = pd.read_csv(ts['csv'])
    df = raw[[ts['file_col'], ts['label_col']]].dropna()
    df = df[df[ts['label_col']].isin(ts['label_map'])].copy()
    idx = build_index(ts['image_dirs'])
    df['path'] = df[ts['file_col']].astype(str).map(idx)
    n_missing = df['path'].isna().sum()
    df = df.dropna(subset=['path']).reset_index(drop=True)
    df['y'] = df[ts['label_col']].map(ts['label_map']).astype(int)
    ts['df'] = df
    nfeed = int((df['y']==0).sum()); nnon = int((df['y']==1).sum()); tot = len(df)
    base = max(nfeed, nnon)/tot
    print(f"[{ts['key']}] {ts['name']}")
    print(f"   resolved images: {tot}  (labelled rows in CSV: {len(raw)}, missing on disk: {n_missing})")
    print(f"   Feeding: {nfeed} ({nfeed/tot:.1%})   Non-feeding: {nnon} ({nnon/tot:.1%})")
    print(f"   always-majority-class baseline accuracy: {base:.1%}\n")

## 4. Load images once per truth set (cached uint8)

Each set's images are decoded once into a `uint8` array and reused for every model. ResNet-50 preprocessing
is applied per batch at prediction time (exactly as in training).

In [ ]:
def load_uint8(path):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    return tf.keras.utils.img_to_array(img).astype('uint8')

for ts in TRUTH_SETS:
    df = ts['df']; N = len(df)
    X = np.zeros((N,)+IMG_SIZE+(3,), dtype='uint8'); bad = []
    t0 = time.time()
    for i, p in enumerate(df['path'].values):
        try: X[i] = load_uint8(p)
        except Exception as e: bad.append((p, str(e)))
        if (i+1) % 1000 == 0: print(f"   [{ts['key']}] {i+1}/{N}  ({time.time()-t0:.0f}s)")
    ts['X'] = X; ts['y'] = df['y'].values.astype(int)
    print(f"[{ts['key']}] loaded {N} images in {time.time()-t0:.0f}s; {len(bad)} failed\n")

## 5. Evaluate every model against both truth sets

Each unique model is loaded **once** and scored on both sets. Metric columns are prefixed by set key
(`gold_*`, `bimby2024_*`). A model scoring < 0.5 on a set has its predictions flipped for that set and
flagged. Models that fail to load are recorded with the error, not dropped.

In [ ]:
def predict_scores(model, X, batch=BATCH):
    out = []
    for i in range(0, len(X), batch):
        xb = preprocess_input(X[i:i+batch].astype('float32'))
        out.append(model.predict(xb, verbose=0).ravel())
    return np.concatenate(out)

def metrics_for(y, scores):
    pred = (scores > THRESHOLD).astype(int)
    acc = accuracy_score(y, pred); flipped = False
    if acc < 0.5:
        pred = 1 - pred; acc = accuracy_score(y, pred); flipped = True
    prec, rec, f1, sup = precision_recall_fscore_support(y, pred, labels=[0,1], zero_division=0)
    return {'acc': round(acc,4),
            'macro_f1': round(f1_score(y, pred, average='macro', zero_division=0),4),
            'feed_recall': round(rec[0],4), 'nonfeed_recall': round(rec[1],4),
            'feed_precision': round(prec[0],4), 'label_flip': flipped,
            'cm': confusion_matrix(y, pred, labels=[0,1]).tolist()}

def evaluate_model(path):
    model = load_model(path, compile=False)
    per_set = {}
    for ts in TRUTH_SETS:
        m = metrics_for(ts['y'], predict_scores(model, ts['X']))
        for k, v in m.items():
            per_set[f"{ts['key']}_{k}"] = v
    del model; tf.keras.backend.clear_session()
    per_set['error'] = ''
    return per_set

def empty_metrics(err):
    d = {'error': err}
    for ts in TRUTH_SETS:
        for k in ['acc','macro_f1','feed_recall','nonfeed_recall','feed_precision','label_flip','cm']:
            d[f"{ts['key']}_{k}"] = None
    return d

results, eval_rows = {}, []
for _, r in cat_df.iterrows():
    name = r['file']
    if name in results:
        metrics = dict(results[name]); note = 'reused (duplicate file)'
    else:
        print(f'evaluating {name} ...')
        try: metrics = evaluate_model(r['path'])
        except Exception as e: metrics = empty_metrics(f'{type(e).__name__}: {e}')
        results[name] = metrics; note = ''
    eval_rows.append({**r.to_dict(), **metrics, 'note': note})
eval_df = pd.DataFrame(eval_rows)
print('\nEvaluation complete.')

In [ ]:
# Ranked side-by-side comparison (ranked by the balanced set's macro-F1)
cols = ['file','trained','date','size_mb',
        'gold_acc','gold_macro_f1','gold_feed_recall','gold_nonfeed_recall',
        'bimby2024_acc','bimby2024_macro_f1','bimby2024_feed_recall','bimby2024_nonfeed_recall',
        'label_flip_any','what_varied','error']
eval_df['label_flip_any'] = eval_df[[f'{ts["key"]}_label_flip' for ts in TRUTH_SETS]].any(axis=1)
ranked = eval_df.sort_values(f'{PRIMARY_KEY}_macro_f1', ascending=False, na_position='last')
pd.set_option('display.max_colwidth', 45)
ranked[cols]

## 6. Save the catalogue + results

In [ ]:
metric_cols = [f'{ts["key"]}_{k}' for ts in TRUTH_SETS
               for k in ['acc','macro_f1','feed_recall','nonfeed_recall','feed_precision','label_flip','cm']]
save_cols = (['file','folder','trained','date','size_mb','what_varied','documented_result',
              'duplicate_of_models_copy'] + metric_cols + ['error','note'])
out_csv = RESULTS_DIR / 'model_catalogue_evaluation.csv'
eval_df[save_cols].to_csv(out_csv, index=False)

md_cols = ['file','trained','date','size_mb',
           'gold_acc','gold_macro_f1','gold_nonfeed_recall',
           'bimby2024_acc','bimby2024_macro_f1','bimby2024_nonfeed_recall','what_varied']
out_md = RESULTS_DIR / 'model_catalogue_evaluation.md'
with open(out_md, 'w') as f:
    f.write('# Feeding model catalogue — dual-truth-set evaluation\n\n')
    for ts in TRUTH_SETS:
        df = ts['df']; nfeed=int((df['y']==0).sum()); nnon=int((df['y']==1).sum()); tot=len(df)
        f.write(f"- **{ts['key']}** ({ts['name']}): {tot} imgs, "
                f"Feeding {nfeed} ({nfeed/tot:.1%}) / Non-feeding {nnon} ({nnon/tot:.1%})\n")
    f.write('\n')
    f.write(ranked[md_cols].to_markdown(index=False))
print('saved:', out_csv); print('saved:', out_md)

## 7. How to read this

- **`*_acc` / `*_macro_f1`** — overall correctness on each set. macro-F1 averages the two classes evenly.
- **`*_feed_recall` / `*_nonfeed_recall`** — fraction of true feeding / non-feeding photos each model caught.
  The real-only models historically had high feed_recall but weak nonfeed_recall (they call anything with a
  flower 'feeding'); the real+super models balance the two.
- **Trust `gold_*` for ranking.** It's balanced, so its accuracy is honest. On **`bimby2024`**, ignore accuracy
  (a do-nothing 'always Feeding' model already scores ~63.5% there) and read **`bimby2024_nonfeed_recall`** and
  **`bimby2024_macro_f1`** instead. A model that's strong on `gold` but whose `bimby2024_nonfeed_recall` craters
  is over-predicting feeding.
- **`label_flip_any = True`** — the model scored < 50% on at least one set until flipped, meaning it was trained
  with the opposite class order. Metrics are valid post-flip, but note it before reusing the model.
- **`*_cm`** (in the CSV) — confusion matrix, rows true `[Feeding, Non-feeding]`, cols predicted the same way.
- **⚠ cropped-input `detect*` models** were trained on detectron-cropped butterflies; low scores here reflect the
  input mismatch, not necessarily a weak model.

Sanity check: `REAL_AND_SUPER_Final_feeding_model_unfrozen` should land near **0.85 / 0.85 on `gold`**, matching
`results/Output_feeding_test_on_BIMBY2024.txt`.